# 11 — The Training Loop
### Starter Notebook

**Learning objectives**
- Understand language modelling loss (next-token prediction)
- Implement a training loop from scratch
- Add learning rate warmup + cosine decay
- Add gradient clipping
- Use the library Trainer to train a small model on TinyStories

---

In [ ]:
import sys, os, math
sys.path.insert(0, os.path.abspath('../..'))
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Part A — Language Modelling Loss

We train the model to predict the **next token** at every position. This is standard **cross-entropy** over a shifted sequence:

- Input:  `[t_0, t_1, t_2, ..., t_{n-1}]`
- Target: `[t_1, t_2, t_3, ..., t_n]`

### Exercise A1 — Implement the LM loss

In [ ]:
def lm_loss(
    logits:  torch.Tensor,   # (batch, seq_len, vocab_size)
    targets: torch.Tensor,   # (batch, seq_len) — already shifted
    ignore_index: int = -100,
) -> torch.Tensor:
    """
    Cross-entropy loss for language modelling.

    Reshape logits to (batch*seq, vocab) and targets to (batch*seq,)
    then compute F.cross_entropy.
    """
    # TODO: reshape and compute cross-entropy
    pass


# Test: logits from a tiny model
vocab = 50
batch, seq = 4, 16
logits_test  = torch.randn(batch, seq, vocab)
targets_test = torch.randint(0, vocab, (batch, seq))

loss = lm_loss(logits_test, targets_test)
if loss is not None:
    print(f'LM Loss: {loss.item():.4f}')
    print(f'Expected ~log(vocab) = {math.log(vocab):.4f}  (random model)')
else:
    print('Not implemented yet.')

## Part B — Learning Rate Schedule

Modern LLMs use a **warmup** phase followed by **cosine decay**:

$$\text{lr}(t) = \begin{cases} \frac{t}{T_{warmup}} \cdot \text{lr}_{peak} & t < T_{warmup} \\ \text{lr}_{min} + \frac{1}{2}(\text{lr}_{peak} - \text{lr}_{min})\left(1 + \cos\frac{\pi(t-T_{warmup})}{T_{max}-T_{warmup}}\right) & t \geq T_{warmup} \end{cases}$$

### Exercise B1 — Implement the schedule

In [ ]:
def get_lr(
    step:       int,
    warmup_steps: int,
    max_steps:  int,
    peak_lr:    float = 3e-4,
    min_lr:     float = 1e-5,
) -> float:
    """
    Compute learning rate for the given step.
    """
    # TODO: linear warmup
    if step < warmup_steps:
        pass
    # TODO: cosine decay
    else:
        pass
    return None   # replace with computed lr


max_steps, warmup = 1000, 100
lrs = [get_lr(s, warmup, max_steps) for s in range(max_steps)]

if lrs[0] is not None:
    plt.figure(figsize=(8, 3))
    plt.plot(lrs)
    plt.axvline(warmup, color='r', linestyle='--', label=f'Warmup ends at step {warmup}')
    plt.xlabel('Step'); plt.ylabel('Learning rate')
    plt.title('Warmup + Cosine Decay Schedule')
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
else:
    print('Not implemented yet.')

## Part C — Training Loop from Scratch

### Exercise C1 — Implement `train_step`

In [ ]:
from src.models.transformer import TransformerLM, TransformerConfig

# Tiny model for demonstration
cfg = TransformerConfig(
    vocab_size=64, d_model=64, n_heads=4, n_layers=2,
    d_ff=128, max_seq_len=32, dropout=0.1,
    ffn_type='standard', norm_type='layernorm', pos_encoding='sinusoidal',
)
model = TransformerLM(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)


def train_step(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    batch: torch.Tensor,          # (batch, seq_len) — token ids
    grad_clip: float = 1.0,
) -> float:
    """
    Single gradient update step.

    Returns: loss value as float
    """
    model.train()
    batch = batch.to(device)

    # Input: all tokens except last; Target: all tokens except first
    x      = batch[:, :-1]   # (batch, seq-1)
    target = batch[:, 1:]    # (batch, seq-1)

    # TODO 1: forward pass → logits
    logits = None

    # TODO 2: compute loss using lm_loss
    loss = None

    # TODO 3: backward + gradient clip + optimizer step
    optimizer.zero_grad()
    # loss.backward()
    # torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    # optimizer.step()

    return 0.0 if loss is None else loss.item()


# Quick test
dummy_batch = torch.randint(0, cfg.vocab_size, (4, cfg.max_seq_len))
loss_val = train_step(model, optimizer, dummy_batch)
print(f'Train step loss: {loss_val:.4f}')

### Exercise C2 — Full training loop

In [ ]:
# Generate synthetic data (random tokens — just to verify the loop)
n_samples = 500
data = torch.randint(0, cfg.vocab_size, (n_samples, cfg.max_seq_len))
dataset = TensorDataset(data)
loader  = DataLoader(dataset, batch_size=32, shuffle=True)

model = TransformerLM(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

n_epochs = 5
step = 0
losses = []

for epoch in range(n_epochs):
    epoch_loss = 0.0
    for (batch,) in loader:
        # TODO: call train_step, update lr, record loss
        loss_val = train_step(model, optimizer, batch)
        losses.append(loss_val)
        step += 1
    print(f'Epoch {epoch+1}/{n_epochs}  loss={sum(losses[-len(loader):]) / len(loader):.4f}')

plt.figure(figsize=(8, 3))
plt.plot(losses, alpha=0.6, color='steelblue')
plt.xlabel('Step'); plt.ylabel('Loss')
plt.title('Training Loss (random data — should fluctuate around log(vocab))')
plt.axhline(math.log(cfg.vocab_size), color='r', linestyle='--', label='Chance level')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## Part D — Using the Library Trainer

In [ ]:
from src.training.trainer import Trainer, TrainingConfig

train_cfg = TrainingConfig(
    max_steps=200,
    learning_rate=3e-4,
    warmup_steps=20,
    gradient_clip=1.0,
    log_every_n_steps=50,
)

model2 = TransformerLM(cfg).to(device)
trainer = Trainer(model=model2, config=train_cfg, device=device)

# Build a minimal DataLoader from synthetic data
dataset2 = TensorDataset(data)
loader2   = DataLoader(dataset2, batch_size=32, shuffle=True)

print('Trainer ready. To train, call:')
print('  trainer.train(train_loader=loader2)')
print('\n(Skipping actual training here to keep the notebook fast.)')
print('See scripts/train.py for a complete training run.')

## Summary

**The training recipe for a language model:**

1. **Loss**: cross-entropy on next-token prediction (shift by 1)
2. **Optimizer**: AdamW with weight decay on non-embedding params
3. **LR Schedule**: linear warmup → cosine decay
4. **Gradient clipping**: norm clip at 1.0 to prevent spikes
5. **Mixed precision**: fp16 for 2x throughput on modern GPUs

**Next:** `12_text_generation_starter.ipynb` — generating text from a trained model.